# 第 15 章　分析の統合

2024 年 8 月は，熊谷と福岡のどちらが暑かったのでしょうか。日最高気温と日最低気温のどちらに注目するか，基準に達した日数と同じ日の大小のどちらを比べるかによって，答えは変わります。

このノートでは，読者が知りたいことを決め，数字と図で確かめられる短い気象の記事を作ります。講義では月全体を調べます。実習では月の前半と後半を同じ方法で比べ，前半だけで書いた説明を後半の結果も含む文章へ改めます。

前半は，章本文の説明に合わせて，上から順にセルを実行します。コードの前に何を調べるかを読み，実行後は表や図のどの値が問いへの答えになるかを確かめてください。後半の約 20 分の実習では，比べる対象や条件を自分で選び，結果を短い文章にします。前半のコードは実行できる見本なので，作図の処理をすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "jma-temperature-2024-08.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

temperatures = pd.read_csv(
    csv_path, dtype={"station_code": str}, parse_dates=["date"],
)
station_codes = ["47626", "47807"]
station_labels = ["Kumagaya", "Fukuoka"]
station_colors = ["#195b8a", "#a7421f"]

temperatures[["station", "date", "max_temp_c", "min_temp_c"]].head()

## 本文と一緒に実行

### 1. 「暑かった」を具体的な気温で比べる

まず 8 月 7 日の二地点を比べます。日最高気温が高い地点は，日最低気温も高いのでしょうか。本文にある原表で，地点・日付・気温の項目を確認してから，CSV の対応する二行を選びます。

図では，各地点の一日の最低・平均・最高気温を同じ横軸に置きます。左右の端のどちらを比べるかによって，気温が高い地点が変わるかを確かめてください。

In [ ]:
example_date = pd.Timestamp("2024-08-07")
temperature_fields = ["mean_temp_c", "max_temp_c", "min_temp_c"]
one_date = temperatures.loc[temperatures["date"] == example_date]
one_date[["station", "date"] + temperature_fields]

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 2.7)
fig, ax = plt.subplots(layout="constrained")
for level, code, color in zip([1, 0], station_codes, station_colors):
    day = one_date.loc[one_date["station_code"] == code].iloc[0]
    low, mean, high = day["min_temp_c"], day["mean_temp_c"], day["max_temp_c"]
    ax.hlines(level, low, high, color=color, linewidth=3)
    ax.scatter([low, high], [level, level], color=color, marker="|", s=190)
    ax.scatter(mean, level, color=color, marker="D", s=41, zorder=3)
    for value, name in [(low, "Min"), (mean, "Mean"), (high, "Max")]:
        ax.annotate(f"{name} {value:.1f}", (value, level), xytext=(0, 11),
                    textcoords="offset points", ha="center", color=color, fontsize=9)
ax.set(xlabel="Temperature (°C)", yticks=[1, 0], yticklabels=station_labels,
       xlim=(22.5, 37.5), ylim=(-0.4, 1.6))
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.show()

線の左端は日最低気温，右端は日最高気温，ひし形は日平均気温です。日平均の点が両端の中点になるとは限らない理由を，本文の「どの観測値を平均したか」という説明から考えてください。

日最低気温は，一日全体で最も低かった値です。夜間だけの最低気温を集計した列ではありません。記事に使う言葉も，選んだ列が表す時間の範囲に合わせます。

### 2. 同じ日数を調べてから基準以上の日を数える

日最高気温が 35 ℃以上だった日数を比べるとき，一方の記録だけが欠けていれば，日数差の意味が変わります。まず，二地点とも同じ 31 日分の気温を比べられるかを確認します。

一地点に 31 行あっても，同じ日付が重複していれば，全 31 日が一回ずつあるとは限りません。地点と日付の重複，気温の欠損，8 月の暦との一致を順に調べます。平均気温が最低気温と最高気温の間にあるかも，項目どうしの関係を確かめる手掛かりになります。

In [ ]:
station_day = ["station_code", "date"]
by_station = temperatures.groupby(["station_code", "station"])
coverage = by_station.agg(
    rows=("date", "size"),
    dates=("date", "nunique"),
    maximum_values=("max_temp_c", "count"),
    minimum_values=("min_temp_c", "count"),
    mean_values=("mean_temp_c", "count"),
)
print("Duplicate station-date rows:", temperatures.duplicated(station_day).sum())
coverage

In [ ]:
august = pd.date_range("2024-08-01", "2024-08-31", freq="D")
for (code, station), station_rows in by_station:
    dates = pd.DatetimeIndex(station_rows["date"]).sort_values()
    print(code, station, dates.equals(august))

ordered_temperatures = (
    (temperatures["min_temp_c"] <= temperatures["mean_temp_c"])
    & (temperatures["mean_temp_c"] <= temperatures["max_temp_c"])
)
print("Rows with minimum ≤ mean ≤ maximum:", ordered_temperatures.sum())

### 記録が一日欠けたら

次は 2024 年 8 月 25〜27 日の日最高気温です。**練習のために福岡の 26 日の行を外しています。** 記録がない日を，基準未満の日として数えてよいかを考えます。

| 地点 | 日付 | 日最高気温（℃） |
|---|---|---:|
| 熊谷 | 2024-08-25 | 34.0 |
| 熊谷 | 2024-08-26 | 35.8 |
| 熊谷 | 2024-08-27 | 33.8 |
| 福岡 | 2024-08-25 | 32.4 |
| 福岡 | 2024-08-27 | 37.4 |

1. 一行は何を表しますか。25 日と 27 日に限定すると，各地点で日最高気温が 35 ℃以上だったのは何日ですか。分子と分母を示してください。
2. 二地点の 25〜27 日をすべて比べるには，元の資料のどの値を確認しますか。

（ここに記入）

この小例を確認したら，全 31 日の記録がある実際の表に戻ります。次の二つのセルでは，日最高気温が 35 ℃以上か，日最低気温が 25 ℃以上かを判定し，地点ごとに該当日数と割合を求めます。

In [ ]:
criteria = temperatures.assign(
    maximum_at_least_35=temperatures["max_temp_c"] >= 35,
    minimum_at_least_25=temperatures["min_temp_c"] >= 25,
)
criteria[["station", "date", "max_temp_c", "maximum_at_least_35"]].loc[
    criteria["max_temp_c"] == 35
]

In [ ]:
monthly_counts = criteria.groupby("station_code").agg(
    days=("date", "size"),
    days_max_at_least_35=("maximum_at_least_35", "sum"),
    days_min_at_least_25=("minimum_at_least_25", "sum"),
)
monthly_counts["percent_max_at_least_35"] = (
    100 * monthly_counts["days_max_at_least_35"] / monthly_counts["days"]
)
monthly_counts["percent_min_at_least_25"] = (
    100 * monthly_counts["days_min_at_least_25"] / monthly_counts["days"]
)
monthly_counts.round(1)

日最高気温が 35 ℃以上だったのは熊谷 20 日・福岡 17 日，日最低気温が 25 ℃以上だったのは熊谷 24 日・福岡 30 日です。両地点とも分母は 31 日です。

35.0 ℃ちょうどの日も含む条件が，コードのどの記号で表されているかを確かめてください。記事では，「何℃以上」「どの気温」「何日間」を書くことで，同じ条件で日数を数え直せるようにします。

### 3. 同じ日の二地点を比べる

基準以上の日数だけでは，「同じ日にどちらの地点が高かったか」は分かりません。まず日付順の折れ線を描き，二地点の大小が入れ替わる日を探します。

続いて日付をキーに二地点の表を結合し，「熊谷 − 福岡」の気温差を求めます。後の散布図では，一つの日付について，日最高気温の差を横軸，日最低気温の差を縦軸に置きます。折れ線の一点と，差の散布図の一点が何を表すかを区別してください。

In [ ]:
import matplotlib.dates as mdates
kuma_code, fuku_code = station_codes
rules = [("max_temp_c", 35, "Daily maximum ≥ 35 °C"),
         ("min_temp_c", 25, "Daily minimum ≥ 25 °C")]
plt.rcParams["figure.figsize"] = (7.2, 5.4)
fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, layout="constrained")
for ax, (measure, threshold, title) in zip(axes, rules):
    for code, label, color, style in zip(
        station_codes, station_labels, station_colors, ["-", "--"],
    ):
        station_rows = temperatures.loc[temperatures["station_code"] == code]
        station_rows = station_rows.sort_values("date")
        ax.plot(station_rows["date"], station_rows[measure], label=label,
                color=color, linestyle=style, marker=".", markersize=3.6)
    ax.axhline(threshold, color="#6e7880", linestyle=":", linewidth=1.1)
    ax.set(ylabel="Temperature (°C)", title=title.split(" ≥ ")[0], ylim=(22, 40))
axes[0].legend(frameon=False, ncol=2, loc="lower left")
axes[-1].set_xticks(august[::5])
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
axes[-1].set_xlabel("Date (2024)")
plt.show()

In [ ]:
pair_fields = ["date", "max_temp_c", "min_temp_c"]
kumagaya = temperatures.loc[temperatures["station_code"] == kuma_code, pair_fields]
fukuoka = temperatures.loc[temperatures["station_code"] == fuku_code, pair_fields]

paired = kumagaya.merge(
    fukuoka,
    on="date",
    how="outer",
    validate="one_to_one",
    suffixes=("_kumagaya", "_fukuoka"),
    indicator=True,
).sort_values("date")
paired["_merge"].value_counts()

In [ ]:
paired["maximum_gap_c"] = (
    paired["max_temp_c_kumagaya"] - paired["max_temp_c_fukuoka"]
).round(1)
paired["minimum_gap_c"] = (
    paired["min_temp_c_kumagaya"] - paired["min_temp_c_fukuoka"]
).round(1)
paired.loc[paired["date"] == example_date, ["date", "maximum_gap_c", "minimum_gap_c"]]

In [ ]:
plt.rcParams["figure.figsize"] = (6.5, 4.8)
fig, ax = plt.subplots(layout="constrained")
ax.axvline(0, color="#77838c", linestyle=":", linewidth=1)
ax.axhline(0, color="#77838c", linestyle=":", linewidth=1)
ax.scatter(paired["maximum_gap_c"], paired["minimum_gap_c"], s=30, color="#465f72")
for day_number, offset in [(7, (-67, -16)), (12, (-28, 8)), (17, (-35, 17)), (22, (9, 3))]:
    row = paired.loc[paired["date"].dt.day == day_number].iloc[0]
    coords = (row["maximum_gap_c"], row["minimum_gap_c"])
    ax.scatter(*coords, s=47, facecolor="white", edgecolor=station_colors[0], zorder=3)
    ax.annotate(f"Aug {day_number}", coords, xytext=offset,
                textcoords="offset points", fontsize=9,
                arrowprops={"arrowstyle": "-", "color": "#697a88", "linewidth": 0.7})
ax.set(xlabel="Daily maximum: Kumagaya − Fukuoka (°C)",
       ylabel="Daily minimum: Kumagaya − Fukuoka (°C)",
       xlim=(-6, 6), ylim=(-5.7, 3), xticks=[-6, -4, -2, 0, 2, 4, 6])
plt.show()

In [ ]:
gap_counts = pd.DataFrame({
    "Kumagaya higher": (paired[["maximum_gap_c", "minimum_gap_c"]] > 0).sum(),
    "Equal": (paired[["maximum_gap_c", "minimum_gap_c"]] == 0).sum(),
    "Fukuoka higher": (paired[["maximum_gap_c", "minimum_gap_c"]] < 0).sum(),
})
gap_counts

差の散布図で右下にある点は，その日の最高気温は熊谷が高く，最低気温は福岡が高かったことを表します。差を引く順序と，正・負の意味を確かめてください。

日最低気温が同日の熊谷より高かった福岡の 28 日と，福岡で日最低気温が 25 ℃以上だった 30 日は，何を数えた点で違うでしょうか。両地点とも 25 ℃以上の日でも大小を比べられることを手掛かりに，二つの問いを説明します。

### 4. 二地点で基準に達した日の重なりを調べる

記事に「少なくとも片方の地点で 35 ℃以上になった日数」を書きたいとします。熊谷の 20 日と福岡の 17 日を足せばよいでしょうか。同じ日に両地点が基準以上なら，その日を重複して数えることになります。

日付を対応させた表で，二地点の判定がともに `True`，片方だけ `True`，ともに `False` の四通りを数えます。`pd.crosstab()` の行は熊谷，列は福岡の判定です。

In [ ]:
kuma_reached = paired["max_temp_c_kumagaya"] >= 35
fuku_reached = paired["max_temp_c_fukuoka"] >= 35
pd.crosstab(
    kuma_reached.rename("Kumagaya"),
    fuku_reached.rename("Fukuoka"),
)

表の四つのセルが，それぞれどの条件の日数かを言葉で説明してください。熊谷が `True` のセルを集めると熊谷の 20 日，福岡が `True` のセルを集めると福岡の 17 日へ戻ります。

「少なくとも片方が基準以上だった暦日」を数える場合は，両地点とも基準以上だった日を一回だけ含めます。記事が地点ごとの日数を伝えるのか，該当する暦日の日数を伝えるのかを決めてから，足すセルを選んでください。

### 5. 基準を替えたときも同じ説明になるか

猛暑日は日最高気温 35 ℃以上の日です。その報告では定義を変えません。一方，「さらに高い気温に達した日も熊谷が多かったか」という別の問いには，別の基準で数える必要があります。

地点と期間は 8 月全体のまま，34・35・36 ℃の三つの基準で，同じ関数を使って日数を求めます。続いて，月内の日最高気温の平均・中央値・最大も確認します。どの数字でも地点の順位が同じになるかを調べ，記事の表現をどこまで広げてよいかを考えます。

In [ ]:
def count_days_at_or_above(data, temperature_column, threshold):
    """Count qualifying days for each station."""
    values = data[temperature_column]
    reached = values >= threshold
    return reached.groupby(data["station_code"]).sum()


thresholds = [34, 35, 36]
counts_by_threshold = pd.DataFrame(
    [count_days_at_or_above(temperatures, "max_temp_c", limit) for limit in thresholds],
    index=pd.Index(thresholds, name="threshold_c"),
)
counts_by_threshold

In [ ]:
plt.rcParams["figure.figsize"] = (6.5, 3.7)
fig, ax = plt.subplots(layout="constrained")
for code, label, color, marker, style in zip(
    station_codes, station_labels, station_colors, ["s", "o"], ["-", "--"],
):
    ax.plot(counts_by_threshold.index, counts_by_threshold[code], marker=marker,
            linestyle=style, color=color, label=label)
ax.set(xlabel="Daily maximum temperature threshold (°C)", ylabel="Days at or above",
       xticks=thresholds, ylim=(0, 31), xlim=(33.85, 36.15))
ax.legend(frameon=False)
plt.show()

In [ ]:
monthly_maximum = by_station["max_temp_c"].agg(["mean", "median", "max"])
monthly_maximum.round(2)

基準を 34 ℃から 35 ℃へ上げると，どの温度範囲の日が数える対象から外れるでしょうか。二地点で外れる日数が違えば，該当日数の大小も変わり得ます。

平均・中央値・最大は，基準以上の日数とは違う問いに答える数です。月全体の水準を伝えるのか，最も高かった気温を伝えるのか，ある基準に達した日の多さを伝えるのかを決め，それに合う結果を選んでください。

### 6. 比べ方が伝わる一文を書く

冒頭の「どちらが暑かったか」に，気温の項目と比べ方を示して答えます。講義で調べた結果を一つ選び，地点・期間・気温の項目・数えた条件が分かる一文を書いてください。日数や割合を使うなら，何日分を調べた結果かも添えます。

今回の記録は，この二地点のこの月についてのものです。別の年にも同じ傾向があると述べたい場合に，追加で調べることを一つ挙げます。

（ここに一文と次に調べることを書く）

## Jupyter 実習（約 20 分）

月全体の比較から進み，前半について書いた説明が，後半にも当てはまるかを調べます。**気温の項目と比較の仕方**を，記事の読者が知りたいことに合わせて一つずつ選んでください。比べる期間は全員共通の 1〜15 日と 16〜31 日です。

二つの期間を同じ関数で別々に集計し，結果を並べて記事を書き直します。前半の設定を後半の設定で置き換えるのではなく，一度の全セル実行で両期間の結果が出るようにします。

## 1. 読み手と比較を決める（約 4 分）

記事を読む人に何を伝えたいか，一文で書きます。例えば，一日の高い気温に着目した記事と，一日の最も低い気温にも着目した記事では，読む列が変わります。

| 選ぶもの | 指定する値 | 問いとの対応 |
|---|---|---|
| 気温の項目 | `"max_temp_c"` または `"min_temp_c"` | 日最高気温，または日最低気温を比べる。 |
| 比較の仕方 | `"threshold"` または `"same_day"` | 基準に達した日数，または同じ日に相手の地点より高かった日数を比べる。 |

`threshold` の基準は，日最高気温なら 35 ℃，日最低気温なら 25 ℃です。今回は基準は動かしません。`same_day` では日付で二地点を対応させ，どちらが高かったかを数えます。同じ値の日は『高かった日』に含めません。

例えば二地点とも 35 ℃以上の日は，`threshold` なら両地点に一日ずつ入ります。`same_day` なら，日最高気温が高い方に一日が入ります。この違いを手掛かりに，記事が答える問いを決めてください。

- 読み手に伝えたいこと：
- 選んだ気温の項目と理由：
- 選んだ比較の仕方と，それが読み手の問いに答える理由：
- 選ばなかった比較では，自分の問いへの答えとして何が足りないか：

In [ ]:
selected_measure = None  # "max_temp_c" または "min_temp_c"
comparison_kind = None  # "threshold" または "same_day"


## 2. 二つの期間を同じ方法で比べる（約 8 分）

次の関数は準備済みです。自分で一から作る課題ではありません。選んだ比較がどの処理に対応し，返された数が何を表すかを読みます。

`threshold` は，講義の `count_days_at_or_above()` を使い，基準以上の日を数えます。`same_day` は，日付で二地点を結合し，同じ日の相手地点より高かった日を数えます。どちらも，地点別の該当日数 `count`，比較に使った日数 `days`，割合 `share_pct` を返します。

この月は両地点とも全 31 日の気温があり，欠損がないことを講義で確認しました。まず関数の定義セルを実行し，続くセルで 1〜15 日の結果を求めます。選んだ比較方法に合わせて，`count` が何の日数かを説明してください。

In [ ]:
def compare_temperature_period(records, measure, kind):
    if kind == "threshold":
        threshold = 35 if measure == "max_temp_c" else 25
        counts = count_days_at_or_above(records, measure, threshold)
        days = records.groupby("station_code")[measure].count()
    else:
        first = records.loc[records["station_code"] == station_codes[0], ["date", measure]]
        second = records.loc[records["station_code"] == station_codes[1], ["date", measure]]
        pairs = first.merge(second, on="date", how="inner", validate="one_to_one",
                            suffixes=("_kumagaya", "_fukuoka"))
        first_values = pairs[measure + "_kumagaya"]
        second_values = pairs[measure + "_fukuoka"]
        counts = pd.Series(
            [(first_values > second_values).sum(), (second_values > first_values).sum()],
            index=station_codes,
        )
        days = pd.Series(len(pairs), index=station_codes)
    result = pd.DataFrame({"count": counts, "days": days}).reindex(station_codes)
    result["share_pct"] = 100 * result["count"] / result["days"]
    return result


In [ ]:
valid_measure = selected_measure in ["max_temp_c", "min_temp_c"]
valid_comparison = comparison_kind in ["threshold", "same_day"]
first_result = None
if valid_measure and valid_comparison:
    first_half = temperatures.loc[temperatures["date"].between("2024-08-01", "2024-08-15")]
    first_result = compare_temperature_period(first_half, selected_measure, comparison_kind)
    display(first_result.round(1))
else:
    print("気温の項目と比較の仕方を選んでから，このセルを実行します。")


ここまでに表示されたのは 1〜15 日の結果です。まず，前半だけについて記事の一文を書いてください。どちらの地点が多かったかに加え，何の気温をどう比べ，何日分を調べたかが分かる文にします。

後半でも同じ地点の該当日数が多いと思いますか。理由を添えて予想してから，次のセルで 16〜31 日を調べます。前半の設定と結果も，後で比較できるように残します。

- 前半の結果だけを使った記事の一文：
- 後半についての予想と理由：

In [ ]:
period_results = None
if valid_measure and valid_comparison:
    second_half = temperatures.loc[temperatures["date"].between("2024-08-16", "2024-08-31")]
    second_result = compare_temperature_period(second_half, selected_measure, comparison_kind)
    period_results = pd.concat(
        {"Aug 1–15": first_result, "Aug 16–31": second_result}, names=["period", "station_code"]
    )
    display(period_results.round(1))
    if comparison_kind == "same_day":
        for label, result in [("Aug 1–15", first_result), ("Aug 16–31", second_result)]:
            equal_days = result["days"].iloc[0] - result["count"].sum()
            print(label, "— equal temperature on", int(equal_days), "day(s)")
else:
    print("気温の項目と比較の仕方を選ぶと，両期間の比較表が出ます。")


In [ ]:
if period_results is None:
    print("二つの選択を記入すると，前半と後半の比較図が出ます。")
else:
    measure_title = "Daily maximum" if selected_measure == "max_temp_c" else "Daily minimum"
    if comparison_kind == "threshold":
        threshold = 35 if selected_measure == "max_temp_c" else 25
        comparison_title = f"Days at or above {threshold} °C"
    else:
        comparison_title = "Days warmer than the other station"
    fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(7.2, 3.2),
                             layout="constrained")
    for ax, label in zip(axes, ["Aug 1–15", "Aug 16–31"]):
        result = period_results.loc[label]
        bars = ax.barh(station_labels, result["share_pct"], color=station_colors)
        ax.bar_label(bars, labels=[f"{int(n)}/{int(d)}" for n, d in
                                   zip(result["count"], result["days"])], padding=4)
        ax.set(title=label, xlabel="Share of compared days (%)", xlim=(0, 116),
               xticks=[0, 25, 50, 75, 100])
    axes[0].invert_yaxis()
    fig.suptitle(f"{measure_title}: {comparison_title}")
    plt.show()


表の `Aug 1–15` と `Aug 16–31` を比べます。分母は前半が 15 日，後半が 16 日です。図は両期間とも熊谷，福岡の順なので，同じ地点の割合と，二地点の大小の両方を確かめてください。

集計が自分の問いに対応しているかを確認するため，一つの日付を元の表で調べます。下のセルは 8 月 7 日です。`check_date` を替えると，別の日も選べます。その日の二地点の値を読み，自分の比較方法なら，どちらの地点を該当日として数えるかを説明します。

In [ ]:
check_date = "2024-08-07"
if valid_measure and valid_comparison:
    display(temperatures.loc[temperatures["date"] == check_date,
                             ["station", "date", selected_measure]])


- 確認した日付と二地点の値：
- 自分が選んだ方法では，どちらの地点の日数に入るか：

## 3. 後半も含めて説明を改める（約 8 分）

前半について書いた一文を，後半の結果も伝わる 3〜4 文へ改めます。前半と後半で同じ傾向だった場合も，違った場合も，それぞれの数値を使って説明してください。次の内容を含めます。

1. 誰に何を伝えるために，どの気温・どの比較を選んだか。
2. 前半と後半のそれぞれの該当日数と分母。後半の予想に合った点と異なった点を示し，二地点の違いについて最初の説明をどう保つか，改めるか。
3. この月について分かったことと，別の年にも同じ傾向があると述べるために必要な観測。

（ここに記事を記入）

書いた後は，最初に選んだ問いへ答えているか，数字がどの表や図に対応するか，期間を替えても言える部分と言い直した部分が分かるかを読み返してください。比較方法を増やすこと自体が目的ではありません。自分の問いに必要な結果を選んで説明します。

カーネルを再起動して上から全セルを実行し，**前半・後半の表と図が両方**再び出ることを確かめます。報告の数値をこの表へ戻して照合し，選択理由と記事も保存します。

日最低気温を選んだ場合は，一日全体の最低気温を比較しています。夜間の最低気温を意味する「熱帯夜」の日数としては報告しません。 [気象庁の気温の用語](https://www.jma.go.jp/jma/kishou/know/yougo_hp/kion.html)

<details>
<summary>選択と読み取りの手掛かり</summary>

一日の高い気温に着目するなら `selected_measure = "max_temp_c"`，最も低い気温にも着目するなら `selected_measure = "min_temp_c"` です。

例えば「日最高気温が 35 ℃以上の日がどちらに多いか」という記事には，日最高気温と `comparison_kind = "threshold"` が対応します。「同じ日の日最低気温はどちらで高かったか」なら，日最低気温と `comparison_kind = "same_day"` が対応します。他の組合せも，自分の問いと合っていれば使えます。

`threshold` では両地点に該当する日があるので，二地点の日数を足しても，比べた日数にはなりません。`same_day` では，熊谷が高い日，福岡が高い日，同じ気温の日を合わせると，比べた日数に戻ります。前半と後半は日数が異なるため，図の棒は割合とし，ラベルには分子・分母を添えています。

</details>

## 個人の理解確認

今回の記事では，ある月の全日の記録から日数を数えました。一方，第 9 章と第 10 章では，目的やデータの集まり方に合わせて繰り返し計算を行いました。次の三つの依頼では，どの方法が合うでしょうか。

**今ある値を数える，ブートストラップ，置換検定**から選び，その方法で何を明らかにしたいかを説明してください。繰り返し計算を使う場合は，一回ごとに何を選び直し，何を固定し，得られる一つの数が何を表すかを書きます。使わない場合にも，なぜその問いには手元の記録を数えれば足りるかを説明します。この確認は約 20 分の実習の後，授業の最後に行います。

| 依頼 | データの集まり方と知りたいこと |
|---|---|
| A | 二地点とも 2024 年 8 月の全 31 日の気温がある。その月に日最高気温が 35 ℃以上だった日数を報告したい。 |
| B | 同じ母集団の分布から一回ずつ独立に値を得るという調査の設定で，80 件の体重を得た。この母平均を推定し，推定の幅を示したい。 |
| C | 40 人のうち 20 人を等確率で選んで学習支援を割り当て，残りの 20 人と得点を比べた。誰についても支援で得点が変わらないとしたとき，観測した群の平均差以上の大きさがどれほど出るかを調べたい。 |

- A：選ぶ方法と，その問いに合う理由：
- B：選ぶ方法と，その問いに合う理由：
- C：選ぶ方法と，その問いに合う理由：

回答後は，方法名だけでなく，依頼が知りたい対象と計算で分かることが対応しているかを確かめます。繰り返しの分布を使う場合は，その一つの値が一人の測定値なのか，複数の記録から求めた値なのかも区別してください。

## 出典

気象庁「過去の気象データ検索」の日ごとの値。熊谷と福岡の 2024 年 8 月の観測値から気温列を教材用に抽出・加工した CSV を使用しました。 [熊谷の原表](https://www.data.jma.go.jp/stats/etrn/view/daily_s1.php?prec_no=43&block_no=47626&year=2024&month=8&day=&view=p1) ・ [福岡の原表](https://www.data.jma.go.jp/stats/etrn/view/daily_s1.php?prec_no=82&block_no=47807&year=2024&month=8&day=&view=p1) ・ [気象庁の利用規約](https://www.jma.go.jp/jma/kishou/info/coment.html)